# Notebook 8: Capstone — Resisted Hinge and Three-Scenario Comparison

## Completing the Trilogy: Passive, Assisted, Resisted

In Notebooks 6 and 7 we saw the impedance controller track a lever trajectory with
a **passive hinge** (no external torque) and an **assisted hinge** (proportional torque
helping the controller). Now we reverse the hinge torque to **resist** the controller.

**This completes the three-scenario comparison:** how does the same impedance controller
perform when the environment is neutral, helpful, or hostile?

| Scenario | Hinge motor | Effect on tracking |
|----------|-------------|--------------------|
| Passive (NB 06) | Off | Baseline |
| Assisted (NB 07) | Proportional assist $+\alpha\,\tau_{imp}$ | Improved |
| Resisted (NB 08) | Proportional resist $-\alpha\,\tau_{imp}$ | Degraded |

The **payoff moment**: all three scenarios side by side in a single 2×3 figure.

## Setup (Self-Contained)

In [ ]:
import os
import tempfile
import mujoco
import numpy as np
import matplotlib.pyplot as plt

try:
    import mediapy as media
    HAS_MEDIAPY = True
except ImportError:
    HAS_MEDIAPY = False
    print("mediapy not available — inline rendering disabled")

%matplotlib inline

print(f"MuJoCo version: {mujoco.__version__}")
print(f"NumPy version:  {np.__version__}")

## Load Model and Extract Named Indices

In [ ]:
# Self-contained: load model and extract indices (same as NB 06, 07)
model = mujoco.MjModel.from_xml_path('../models/tendon_capstone.xml')

lever_qpos_idx  = model.joint("lever_hinge").qposadr[0]
lever_vel_idx   = model.joint("lever_hinge").dofadr[0]
block_slide_idx = model.joint("block_slide").qposadr[0]
block_vel_idx   = model.joint("block_slide").dofadr[0]
block_ctrl_idx  = model.actuator("block_motor").id
hinge_ctrl_idx  = model.actuator("hinge_motor").id

tendon_id       = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_TENDON, "cable")
L_natural       = model.tendon_lengthspring[tendon_id, 1]
CABLE_STIFFNESS = model.tendon_stiffness[tendon_id]

print(f"Model loaded: nq={model.nq}, nu={model.nu}, ntendon={model.ntendon}")
print(f"L_natural={L_natural:.4f} m, stiffness={CABLE_STIFFNESS:.1f} N/m")

## Controller (Identical to NB 06, 07 — Same Gains, Same Structure)

In [ ]:
# Same gains and controller as NB 06, 07 — only alpha_assist changes between scenarios
K        = 50.0    # Nm/rad
D        = 10.0    # Nm*s/rad
r_lever  = 0.6     # m
R_EFF    = 0.384   # m  — effective moment arm at theta=0
K_BLOCK  = 2000.0  # N/m
D_BLOCK  = 50.0    # N*s/m


def capstone_controller(model, data, q_des, dq_des, tau_hinge=0.0):
    """Two-level impedance controller (identical to NB 06, 07).

    tau_hinge=0.0 -> passive scenario (NB 06)
    tau_hinge>0.0 -> assisted scenario (NB 07)
    tau_hinge<0.0 -> resisted scenario (NB 08)
    """
    q  = data.qpos[lever_qpos_idx].copy()
    dq = data.qvel[lever_vel_idx].copy()

    tau_imp   = K * (q_des - q) + D * (dq_des - dq)
    tau_grav  = data.qfrc_bias[lever_vel_idx]
    tau_total = tau_imp + tau_grav

    T_desired = max(0.0, tau_total / R_EFF)
    L_desired = T_desired / CABLE_STIFFNESS + L_natural

    tip_x = 1.0 - r_lever * np.cos(q)
    tip_z = 0.5 + r_lever * np.sin(q)
    seg2  = np.sqrt((tip_x - 1.0)**2 + (tip_z - 1.0)**2)

    seg1_desired = L_desired - seg2
    block_x_des  = np.clip(1.0 - seg1_desired, -1.0, 0.5)

    block_x  = data.qpos[block_slide_idx].copy()
    block_xd = data.qvel[block_vel_idx].copy()
    F_block  = K_BLOCK * (block_x_des - block_x) + D_BLOCK * (0.0 - block_xd)

    data.ctrl[block_ctrl_idx] = np.clip(F_block, -100.0, 100.0)
    data.ctrl[hinge_ctrl_idx] = tau_hinge


def get_cable_tension(data, tendon_idx=0):
    """Compute analytical cable tension from current tendon length.

    Note: data.ten_force does not exist in MuJoCo 3.6.0.
    Tension is computed analytically from data.ten_length.
    """
    stretch = float(data.ten_length[tendon_idx]) - L_natural
    return CABLE_STIFFNESS * max(0.0, stretch)


print("Controller ready (same as NB 06, 07).")

## Theory: Resisted Hinge

### Reversing the Assist: Proportional Resistance

In Notebook 7 the hinge applied a *helping* torque:

$$\tau_{hinge} = +\alpha \cdot \tau_{imp} \quad (\alpha > 0, \; \text{assisted})$$

Now we apply an *opposing* torque of the same magnitude:

$$\tau_{hinge} = -\alpha \cdot \tau_{imp} \quad (\alpha > 0, \; \text{resisted})$$

With $\tau_{ext} < 0$ opposing the direction the impedance controller tries to
move the lever, the **effective error increases**. The controller must generate
more tendon tension to overcome the resistance, and if $K$ and $D$ are not large
enough, tracking degrades.

### Why Tracking Degrades

In the passive scenario the cable carries 100% of the required torque. In the
resisted scenario the hinge motor continuously opposes the impedance controller:

$$\tau_{cable} = \tau_{imp} - \tau_{hinge} = \tau_{imp} + \alpha\,|\tau_{imp}| = (1+\alpha)\,\tau_{imp}$$

The cable must carry **more** than 100% of the impedance torque — it also
compensates for the resisting hinge torque. This increases the required cable
stretch, block displacement, and position lag.

This simulates an environment that **fights the controller** — like a stiff joint,
external friction, or an opposing load on the lever.

## Reusable Simulation Function

We define a helper that runs a single scenario and returns the recorded history.
The same function is called three times — for passive, assisted, and resisted —
keeping the comparison exactly fair.

In [ ]:
def run_scenario(alpha_assist, sim_duration=5.0):
    """Run capstone sinusoidal tracking experiment for a given assist fraction.

    Args:
        alpha_assist: hinge assist fraction.
                      0.0 = passive (no hinge torque)
                      +0.4 = assisted (hinge helps controller)
                      -0.4 = resisted (hinge opposes controller)
        sim_duration: simulation time in seconds

    Returns:
        t_hist, q_hist, qd_hist, F_hist, rms_deg
    """
    A_des   = 0.3             # rad (~17 degrees)
    f_des   = 0.5             # Hz
    omega_d = 2 * np.pi * f_des
    dt      = model.opt.timestep
    n_steps = int(sim_duration / dt)

    # Each scenario gets a fresh MjData (no state leaks between runs)
    local_data = mujoco.MjData(model)
    mujoco.mj_forward(model, local_data)

    t_hist  = np.zeros(n_steps)
    q_hist  = np.zeros(n_steps)
    qd_hist = np.zeros(n_steps)
    F_hist  = np.zeros(n_steps)

    for i in range(n_steps):
        t      = local_data.time
        q_des  = A_des * np.sin(omega_d * t)
        dq_des = A_des * omega_d * np.cos(omega_d * t)

        # Record BEFORE stepping
        t_hist[i]  = t
        q_hist[i]  = local_data.qpos[lever_qpos_idx].copy()
        qd_hist[i] = q_des
        F_hist[i]  = get_cable_tension(local_data)

        # Compute tau_imp for proportional hinge torque
        q  = local_data.qpos[lever_qpos_idx].copy()
        dq = local_data.qvel[lever_vel_idx].copy()
        tau_imp   = K * (q_des - q) + D * (dq_des - dq)
        tau_hinge = np.clip(alpha_assist * tau_imp, -10.0, 10.0)

        capstone_controller(model, local_data, q_des, dq_des, tau_hinge=tau_hinge)
        mujoco.mj_step(model, local_data)

    rms_deg = np.degrees(np.sqrt(np.mean((q_hist - qd_hist)**2)))
    return t_hist, q_hist, qd_hist, F_hist, rms_deg


print("Reusable run_scenario() helper ready.")
print("  alpha_assist = 0.0  -> passive")
print("  alpha_assist = +0.4 -> assisted")
print("  alpha_assist = -0.4 -> resisted")

## Experiment 1: Resisted Hinge Tracking

The hinge motor applies a **negative** proportional torque, opposing the
impedance controller at every timestep.

In [ ]:
ALPHA_RESIST = -0.4   # negative: hinge opposes the controller

t_r, q_r, qd_r, F_r, rms_resisted = run_scenario(alpha_assist=ALPHA_RESIST)

print(f"Resisted scenario (alpha={ALPHA_RESIST}):")
print(f"  RMS tracking error: {rms_resisted:.2f} degrees")
print(f"  Lever range: [{np.degrees(q_r.min()):.1f}, {np.degrees(q_r.max()):.1f}] degrees")
print(f"  Max cable tension: {F_r.max():.2f} N")

## Experiment 2: Run All Three Scenarios

Re-run passive and assisted within this notebook (self-contained per the
notebook independence principle). All three scenarios use exactly the same
`run_scenario()` function — only the `alpha_assist` value changes.

In [ ]:
ALPHA_PASSIVE  =  0.0
ALPHA_ASSIST   = +0.4
ALPHA_RESIST   = -0.4

print("Running all three scenarios...")
t_p, q_p, qd_p, F_p, rms_passive  = run_scenario(alpha_assist=ALPHA_PASSIVE)
print(f"  Passive  (alpha= 0.0): RMS = {rms_passive:.2f} deg")

t_a, q_a, qd_a, F_a, rms_assisted = run_scenario(alpha_assist=ALPHA_ASSIST)
print(f"  Assisted (alpha=+0.4): RMS = {rms_assisted:.2f} deg")

t_r, q_r, qd_r, F_r, rms_resisted = run_scenario(alpha_assist=ALPHA_RESIST)
print(f"  Resisted (alpha=-0.4): RMS = {rms_resisted:.2f} deg")

## RMS Comparison Table

In [ ]:
# Compute improvement and degradation
improvement_pct = (rms_passive - rms_assisted) / rms_passive * 100.0
degradation_pct = (rms_resisted - rms_passive) / rms_passive * 100.0

print("\nScenario    | alpha | RMS Error (deg) | vs Passive")
print("------------|-------|-----------------|-------------------")
print(f"Passive     |  0.0  |     {rms_passive:5.2f}       | baseline")
print(f"Assisted    | +0.4  |     {rms_assisted:5.2f}       | {improvement_pct:+.1f}% (better)")
print(f"Resisted    | -0.4  |     {rms_resisted:5.2f}       | {degradation_pct:+.1f}% (worse)")

## Validation

In [ ]:
# Validate ordering: assisted < passive < resisted
assert rms_assisted < rms_passive, (
    f"Assisted RMS ({rms_assisted:.2f}°) must be less than passive ({rms_passive:.2f}°)"
)
assert rms_resisted > rms_passive, (
    f"Resisted RMS ({rms_resisted:.2f}°) must be greater than passive ({rms_passive:.2f}°)"
)

# Validate minimum thresholds (>20% in both directions)
assert improvement_pct > 20.0, (
    f"Assisted improvement {improvement_pct:.1f}% is below 20% threshold."
)
assert degradation_pct > 20.0, (
    f"Resisted degradation {degradation_pct:.1f}% is below 20% threshold."
)

print(f"PASS: Ordering confirmed — assisted ({rms_assisted:.2f}°) < passive ({rms_passive:.2f}°) < resisted ({rms_resisted:.2f}°)")
print(f"PASS: Assisted improvement {improvement_pct:.1f}% > 20% threshold")
print(f"PASS: Resisted degradation {degradation_pct:.1f}% > 20% threshold")

## Three-Scenario Comparison Figure

The culminating visualization: all three scenarios side by side.

- **Top row:** Lever angle tracking — desired (black dashed) vs actual (colored)
- **Bottom row:** Cable tension for each scenario
- **Columns:** Passive | Assisted | Resisted

The visually distinct trajectory curves tell the whole story at a glance:
- Assisted curves hug the desired trajectory (small gap)
- Passive curves show moderate lag
- Resisted curves show the largest departure from the desired path

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey='row')

scenarios = [
    ("Passive",  t_p, q_p, qd_p, F_p, rms_passive,  'b',       ALPHA_PASSIVE),
    ("Assisted", t_a, q_a, qd_a, F_a, rms_assisted, 'g',       ALPHA_ASSIST),
    ("Resisted", t_r, q_r, qd_r, F_r, rms_resisted, 'darkorange', ALPHA_RESIST),
]

for col, (name, t, q, qd, F, rms, color, alpha) in enumerate(scenarios):
    ax_angle  = axes[0, col]
    ax_tension = axes[1, col]

    # --- Top row: angle tracking ---
    ax_angle.plot(t, np.degrees(qd), 'k--', lw=1.5, label='Desired')
    ax_angle.plot(t, np.degrees(q),  color=color, lw=1.5, label='Actual')
    ax_angle.set_title(f"{name} — Lever Angle Tracking", fontsize=11)
    ax_angle.grid(True, alpha=0.3)
    ax_angle.legend(fontsize=9, loc='upper right')
    ax_angle.annotate(
        f'RMS = {rms:.2f}°',
        xy=(0.05, 0.88), xycoords='axes fraction',
        fontsize=10, color=color,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=color)
    )
    if col == 0:
        ax_angle.set_ylabel('Angle (deg)', fontsize=11)

    # --- Bottom row: cable tension ---
    ax_tension.plot(t, F, color=color, lw=1.5)
    ax_tension.fill_between(t, F, 0, alpha=0.15, color=color)
    ax_tension.set_title(f"{name} — Cable Tension", fontsize=11)
    ax_tension.set_xlabel('Time (s)', fontsize=10)
    ax_tension.grid(True, alpha=0.3)
    if col == 0:
        ax_tension.set_ylabel('Tension (N)', fontsize=11)

plt.suptitle(
    "Impedance Control: Passive vs Assisted vs Resisted Hinge",
    fontsize=14, fontweight='bold'
)
plt.tight_layout(rect=[0, 0, 1, 0.95])

save_path = os.path.join(tempfile.gettempdir(), 'nb8_three_scenario_comparison.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Figure saved to {save_path}")

## Inline Video: Resisted Hinge Tracking

The video below shows the resisted scenario in 3D. The hinge motor opposes the impedance controller with 40% of the impedance torque, forcing the cable to carry more than 100% of the required torque. You should see the lever lagging behind the desired trajectory more than in the passive or assisted cases, and the block moving more aggressively to compensate.

Tendon rendering is enabled so the cable path through the corner pulley is visible.

In [ ]:
# --- Inline Video: Resisted hinge impedance tracking ---
try:
    renderer = mujoco.Renderer(model, height=360, width=480)

    # Enable tendon rendering in scene options
    scene_opt = mujoco.MjvOption()
    scene_opt.flags[mujoco.mjtVisFlag.mjVIS_TENDON] = True

    # Trajectory parameters (same as experiment above)
    A_vid   = 0.3
    f_vid   = 0.5
    omega_vid = 2 * np.pi * f_vid
    dt_vid  = model.opt.timestep

    # Reset simulation with fresh data
    vid_data = mujoco.MjData(model)
    mujoco.mj_forward(model, vid_data)

    frames = []
    frame_every = 10
    duration_vid = 5.0
    n_vid = int(duration_vid / dt_vid)

    for i in range(n_vid):
        t = vid_data.time
        q_des  = A_vid * np.sin(omega_vid * t)
        dq_des = A_vid * omega_vid * np.cos(omega_vid * t)

        # Compute resisted hinge torque (negative alpha opposes the controller)
        q  = vid_data.qpos[lever_qpos_idx].copy()
        dq = vid_data.qvel[lever_vel_idx].copy()
        tau_imp   = K * (q_des - q) + D * (dq_des - dq)
        tau_hinge = np.clip(ALPHA_RESIST * tau_imp, -10.0, 10.0)

        capstone_controller(model, vid_data, q_des, dq_des, tau_hinge=tau_hinge)
        mujoco.mj_step(model, vid_data)
        if i % frame_every == 0:
            renderer.update_scene(vid_data, scene_option=scene_opt)
            frames.append(renderer.render())

    renderer.close()
    fps = int(1.0 / (dt_vid * frame_every))

    if HAS_MEDIAPY:
        media.show_video(frames, fps=fps)
    print(f'Video: {len(frames)} frames at {fps} fps')
except Exception as e:
    print(f'Renderer not available in this environment: {e}')
    print('This is expected when running headless (e.g., nbconvert).')

## Optional: Interactive Passive Viewer

In [ ]:
# Uncomment to open interactive viewer (non-blocking on Windows).
# Requires `pip install mujoco` with viewer support.

# view_data = mujoco.MjData(model)
# mujoco.viewer.launch_passive(model, view_data)

## Series Conclusion: Eight Notebooks, One Insight

### What the Comparison Teaches

The 2×3 figure above tells the complete story of impedance control and environmental
interaction. The **same controller gains** ($K=50$ Nm/rad, $D=10$ Nm·s/rad) produce
three measurably different outcomes depending on what the hinge motor does:

1. **Impedance control allows compliant interaction with the environment.**  
   The controller does not rigidly enforce the trajectory; it applies a restoring
   force proportional to the error. External forces change the effective stiffness
   and damping seen at the lever.

2. **The same controller gains produce different outcomes depending on environmental forces.**  
   Tuning gains for one environment does not guarantee performance in another.
   A robot picking up a heavy load (resisted) needs different gains than one
   moving freely through air (passive).

3. **Assisted environments improve tracking; resisted environments degrade it.**  
   When environment and controller work together, errors cancel. When they oppose
   each other, errors amplify. This is the physical meaning of impedance mismatch.

4. **Tendon-driven systems transmit force indirectly.**  
   The controller actuates the block, not the lever. Cable tension is a consequence
   of block position; lever torque is a consequence of cable tension. Understanding
   this indirect chain — and compensating gravity within it — is what makes the
   controller work.

---

### The Arc of Eight Notebooks

Over these 8 notebooks you have built impedance controllers from the simplest
1-DOF mass-spring-damper to a tendon-driven pulley-lever system.

| NB | System | New concept |
|----|--------|-------------|
| 1 | Mass-spring-damper (1-DOF) | Impedance law, gain-timestep stability |
| 2 | Pendulum | Gravity compensation, nonlinear geometry |
| 3 | 2-DOF planar arm | Multi-DOF coupling, joint-space impedance |
| 4 | Cartesian impedance | Task-space vs joint-space control |
| 5 | Tendon API intro | Cable springs, tension-only mechanics |
| 6 | Capstone passive | Two-level controller, cable-driven lever |
| 7 | Capstone assisted | Environmental assist, proportional sharing |
| 8 | Capstone resisted | Environmental resistance, degradation analysis |

The **core insight is always the same**:

$$\tau = K(\theta_d - \theta) + D(\dot{\theta}_d - \dot{\theta})$$

What changes is how the system geometry and environment mediate between your control
effort and the controlled variable. Gravity compensation, cable inverse kinematics,
and environmental torques all reshape the effective impedance — but the fundamental
law remains a spring and a damper tracking a desired trajectory.